# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Login successful")

Login successful


In [24]:
from huggingface_hub import HfApi

api = HfApi()
info = api.dataset_info("FlyRank/internship-warehouse")
print([s.rfilename for s in info.siblings])

['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet', 'fact_content_daily_performance/month=2025-07/data_0.parquet', 'fact_content_daily_performance/month=2025-08/data_0.parquet', 'fact_content_daily_performance/month=2025-09/data_0.parquet', 'fact_content_daily_performance/month=2025-10/data_0.parquet', 'fact_content_daily_performance/month=2025-11/data_0.parquet', 'fact_content_daily_performance/month=2025-12/data_0.parquet', 'fact_content_daily_performance/month=2026-01/data_0.parquet', 'fact_content_daily_performance/month=2026-02/data_0.parquet', 'fact_content_daily_performance/month=20

In [25]:
files = [s.rfilename for s in info.siblings]
prefixes = sorted(set(f.split('/')[0] for f in files))
for p in prefixes:
    print(p)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance
fact_content_daily_performance_sample.parquet
fact_content_query_90d.parquet


In [26]:
from datasets import load_dataset

ds = load_dataset("FlyRank/internship-warehouse", "dim_clients")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start'],
        num_rows: 104
    })
})


In [27]:
ds_perf = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
print(ds_perf)
print(ds_perf["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 9841378
    })
})
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', '

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [28]:
# Unit of analysis: One row = one content item's performance for one client, on one specific day (report_date + client_hash_id + content_hash_id).
# Time window: month = 2026-03 (mid-panel month, March 2026).

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [30]:
print(ds_perf["train"].column_names)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [31]:
# Feature examples: gsc_impressions, gsc_clicks (or similar performance metrics), client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available
# Label/proxy: next-day or next-period performance metric (e.g. gsc_clicks) for the same client+content — what we want to predict or rank
# Context (not a feature): report_date, client_hash_id, content_hash_id — identify the row but aren't fed as model inputs directly
# Excluded on purpose: fact_content_daily_performance_sample.parquet (the sealed June 2026 test month) — never used to build or tune the label logic

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [33]:
df = ds_perf["train"].to_pandas()
print(df.shape)
print(df.dtypes)

(9841378, 30)
report_date                  object
client_hash_id               object
content_hash_id              object
client_has_gsc                 bool
client_has_ga4                 bool
gsc_data_available             bool
ga4_data_available           object
gsc_impressions               int64
gsc_clicks                    int64
gsc_sum_position              int64
gsc_avg_position            float64
ga4_pageviews               float64
ga4_sessions                float64
ga4_users                   float64
ga4_engaged_sessions        float64
ga4_total_engagement_sec    float64
sessions_organic            float64
sessions_direct             float64
sessions_referral           float64
sessions_social             float64
sessions_paid               float64
sessions_ai                 float64
ai_chatgpt                  float64
ai_perplexity               float64
ai_gemini                   float64
ai_copilot                  float64
ai_claude                   float64
ai_meta       

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [35]:
grain_check = df.duplicated(subset=["report_date", "client_hash_id", "content_hash_id"]).sum()
print("Duplicate rows on grain:", grain_check)
print("Total rows:", len(df))

Duplicate rows on grain: 0
Total rows: 9841378


In [36]:
print("Row count:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())
print("Unique dates:", df["report_date"].nunique())

Row count: 9841378
Date range: 2026-03-01 to 2026-03-31
Unique dates: 31


In [37]:
gsc_available = df.query("gsc_data_available == True")
print("Rows with gsc_data_available IS TRUE:", len(gsc_available))
print("Out of total rows:", len(df))
print("Percentage available:", round(len(gsc_available) / len(df) * 100, 2), "%")

Rows with gsc_data_available IS TRUE: 3611061
Out of total rows: 9841378
Percentage available: 36.69 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [38]:
# Limitation: This slice only covers clients with client_has_gsc/client_has_ga4 flags set at query time —
# it can't tell us about clients before they connected GSC/GA4, or about content performance outside this
# single month's window (no history before or after March 2026 is visible here).

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [40]:
features = df[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]].copy()

print(features.head())
print(features.shape)

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  \
0               20           0          3.350000           NaN   
1                1           0          0.000000           NaN   
2              125           1          4.928000           NaN   
3                7           0          4.000000           NaN   
4               11           0          2.272727           NaN   

   ga4_engaged_sessions  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN  
(9841378, 5)


In [41]:
# Feature "available when?" notes:
# gsc_impressions - known at decision moment because it's logged by GSC for that same report_date, before the next day starts
# gsc_clicks - same as above, recorded for that day only, available same-day
# gsc_avg_position - computed from that day's search data, available same-day
# ga4_sessions - logged by GA4 for that day, available same-day
# ga4_engaged_sessions - logged by GA4 for that day, available same-day

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Use a smaller sample to avoid memory crash
sample_df = df.sample(n=200000, random_state=42).sort_values(["client_hash_id", "content_hash_id", "report_date"])
sample_df["next_day_clicks"] = sample_df.groupby(["client_hash_id", "content_hash_id"])["gsc_clicks"].shift(-1)
sample_valid = sample_df.dropna(subset=["next_day_clicks"])

X = sample_valid[["gsc_impressions", "gsc_avg_position", "ga4_sessions", "ga4_engaged_sessions"]].fillna(0)
y_future = sample_valid["next_day_clicks"]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y_future, test_size=0.2, random_state=42)
model2 = LinearRegression().fit(X_train2, y_train2)
preds2 = model2.predict(X_test2)
print("HONEST R2 (predicting NEXT day's clicks, no leak):", r2_score(y_test2, preds2))

X_leak2 = X.copy()
X_leak2["gsc_clicks_today"] = sample_valid["gsc_clicks"].values

X_train3, X_test3, y_train3, y_test3 = train_test_split(X_leak2, y_future, test_size=0.2, random_state=42)
model3 = LinearRegression().fit(X_train3, y_train3)
preds3 = model3.predict(X_test3)
print("LEAKY R2 (using today's clicks to predict next day):", r2_score(y_test3, preds3))

HONEST R2 (predicting NEXT day's clicks, no leak): 0.31126300192382583
LEAKY R2 (using today's clicks to predict next day): 0.3771705897644342


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.